# Reproducing Park et al. (2019) from CRCNS hc-20

**Paper.** Park EH, Keeley S, Savin C, Ranck JB Jr, Fenton AA (2019). How the internally organized
direction sense is used to navigate. *Neuron* 101:285-293.
[doi:10.1016/j.neuron.2018.11.019](https://doi.org/10.1016/j.neuron.2018.11.019). The claims file
(`claims/park-2019.yaml`) was written from the version of record with its STAR Methods and
Supplemental Information, and from the paper's code
(https://github.com/FentonLab/PyCode_MEC_HDC-NeuronMS_Park at commit 545a738).

**Data set.** CRCNS hc-20, [doi:10.6080/K0SF2TDN](https://doi.org/10.6080/K0SF2TDN), the file
`hc-20/live/data.zip` in the public S3 bucket `crcnsarchive`, streamed and opened in place. CRCNS
data sets are not versioned, and the paper cites the data set's DOI only, so the version pinned
is the one published now, recorded as `unversioned` with the MD5 sum of `data.zip` from the data
set's `checksums.md5` (`b82c7b97b0ba3491f04ecc4dbcd48e83`); the setup cell below checks it
against the published file, so a later change is detectable. There is no other version to
choose between. The data descriptor is version 0.6 (14 Nov 2018).

**What the data set holds.** The spike trains and 30 Hz tracking of the 115 head-direction cells
in the three trials (stable1, rotating in room and arena coordinates, stable2), a file list that
matches units across trials, and the tracker's files of each trial. It holds no other MEC
units, no LFP, no waveforms and no training sessions.

| Claim | Tested here |
|---|---|
| cell-counts | tested |
| hd-cell-rate | tested |
| fig1a-avoidance-learning | blocked, data missing (training sessions not deposited) |
| fig1c-speed | tested |
| fig1c-lfp-power | blocked, data missing (no LFP) |
| fig1c-firing-rate | tested |
| fig1c-directionality | tested |
| fig1d-preference-change | tested |
| fig1d-instability | tested |
| fig2-decoding-stable-vs-rotating | tested |
| figs2-frame-preference | tested |
| fig2c-proximity-anticorrelation | tested |
| figs3-no-head-direction-bias | tested |
| fig3-hd-pair-correlations-preserved | tested |
| figs4-all-pair-correlations-preserved | blocked, data missing (only head-direction cells deposited) |
| figs5-hd-cells-high-model-information | blocked, data missing (the 921 modelled cells are not deposited) |
| figs6-subclass-correlations-preserved | blocked, data missing (as figs5) |
| no-angular-velocity-tuning | tested |
| fig4-internal-tuning-preserved | tested |
| fig4c-arena-registration-short-lived | tested |
| fig5-error-inhomogeneous | tested |
| fig5c-error-minima-at-cues | tested |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import hd20

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
started = time.time()

sums = hd20.checksums()
print("checksums.md5 of hc-20/live:", sums)
print("data.zip MD5 as pinned:", hd20.DATA_ZIP_MD5, "; as published now:", sums["data.zip"],
      "; same:", sums["data.zip"] == hd20.DATA_ZIP_MD5)
units, data = hd20.load_units()
print(f"loaded {len(units)} units x {len(hd20.TRIALS)} trials from {hd20.DATASET} ({hd20.DATASET_VERSION}) "
      f"in {time.time() - started:.0f} s")
TR = hd20.TRIALS
U = list(units.unit)
SESS = units.groupby("session").unit.apply(list).to_dict()

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
rows = []
for u in U:
    for tr in TR:
        x = data[u][tr]
        ok = x["d"] <= 360
        rows.append({"unit": u, "trial": tr, "fs_header": x["fs_header"], "fs": x["fs"], "samples": len(x["d"]),
                     "duration_s": len(x["d"]) / x["fs"], "spikes": int(x["n"].sum()),
                     "untracked_dir": 1 - ok.mean(), "xy_zero": np.mean((x["x"] == 0) & (x["y"] == 0)),
                     "dir_min": x["d"][ok].min(), "dir_max": x["d"][ok].max(),
                     "x_max": x["x"].max(), "y_max": x["y"].max(), "max_spk_per_sample": x["n"].max(),
                     "spikes_sorted": bool(np.all(np.diff(x["spike_t"]) >= 0)), "trailing_bytes": x["trailing"]})
survey = pd.DataFrame(rows)
TRIALDUR = survey.set_index(["unit", "trial"]).duration_s
print("Units:", len(U), "; sessions (date and rat):", units.session.nunique(), "; rats:", units.rat.nunique())
print("\nUnits per rat:")
print(units.groupby("rat").agg(units=("unit", "size"), sessions=("session", "nunique")).T.to_string())
print("\nPer trial, over units:")
print(survey.groupby("trial")[["fs", "samples", "duration_s", "spikes", "untracked_dir", "xy_zero", "dir_min",
                                "dir_max", "x_max", "y_max", "max_spk_per_sample"]]
      .agg(["min", "median", "max"]).T.round(3).to_string())
print("\nHeaders with a sampling rate other than about 30 Hz:")
print(survey[survey.fs_header > 100][["unit", "trial", "fs_header", "fs", "samples"]].to_string(index=False))
print("\nFiles shorter than 29 min:")
print(survey[survey.duration_s < 29 * 60][["unit", "trial", "samples", "duration_s", "spikes"]].to_string(index=False))
print("\nFiles whose spike times are not in order:")
for _, r in survey[~survey.spikes_sorted].iterrows():
    dd = np.diff(data[r.unit][r.trial]["spike_t"])
    print(f"  {r.unit} {r.trial}: {int((dd < 0).sum())} decreasing step(s) of {dd.min() / 10:.1f} ms in {len(dd) + 1} spikes")
print("\nTrailing bytes after the last whole sample in any file:", int(survey.trailing_bytes.max()))
print("Samples with more than 15 spikes (in 33 ms):")
for u in U:
    for tr in ["SA1", "RA_rm", "SA2"]:
        n = data[u][tr]["n"]
        if n.max() > 15:
            print(f"  {u} {tr}: {int((n > 15).sum())} samples, max {n.max()}")

The data descriptor says the TSD files sample position and direction at 30 Hz, with times in
100 us units and directions in degrees; the headers give 30.008 to 30.023 samples/s, and 30 min
trials hold about 54,000 samples (1798 s), consistent with it. One header (1017R035-P11C2,
stable2) states 2000 samples/s and a first timestamp of 5; the file holds 54,185 samples, a
30 min trial at 30 Hz, so its rate is taken as 30 Hz. Directions run from 0 to 359 degrees;
directions above 360 (4294967295, that is -1) mark samples without a direction, 4 to 73% of a
file's samples, which the descriptor does not mention. Positions are pixels from 0 to 255; x =
y = 0 marks lost positions. The rotating trial's room and arena files of a unit hold the same
spikes (checked below).

In [ ]:
fl = units[["unit", "session"] + [f"listed_{t}" for t in TR] + [f"file_{t}" for t in TR]]
mism = fl[fl[[f"listed_{t}" for t in TR]].to_numpy() != fl[[f"file_{t}" for t in TR]].to_numpy()].dropna(how="all")
shifted = units[units.listed_SA2 != units.file_SA2]
print("Rows of FileNameList.xlsx whose files are not all the same tetrode and cluster:")
print(shifted[["unit", "file_SA1", "listed_SA2", "file_SA2"]].to_string(index=False))
print("\nSpike counts of these units (by file name) in each trial, and of the SA2 file the list gives:")
for _, r in shifted.iterrows():
    listed = [k for k in U if units.set_index("unit").loc[k, "file_SA2"] == r.listed_SA2][0]
    print(f"  {r.unit}: SA1 {data[r.unit]['SA1']['n'].sum()}, RA {data[r.unit]['RA_rm']['n'].sum()}, "
          f"SA2 by name {data[r.unit]['SA2']['n'].sum()}, SA2 as listed {data[listed]['SA2']['n'].sum()}")
print("\nRoom and arena files of the rotating trial hold the same spikes for every unit:",
      all(np.array_equal(data[u]["RA_rm"]["n"], data[u]["RA_ar"]["n"]) for u in U))
same = []
for s, us in SESS.items():
    for tr in TR:
        ref = data[us[0]][tr]
        for u in us[1:]:
            same.append(np.array_equal(data[u][tr]["d"][:26954], ref["d"][:26954]))
print("Cells of a session share their tracking in every trial:", all(same), f"({len(same)} comparisons)")

The data descriptor says FileNameList.xlsx lists the files of one unit on one row. For session
1121R157 the stable2 column is shifted by one row against the other three for five rows (each
unit's stable2 file is the next unit's), and the last of them takes the file `cl1.P11C2`. Units
are matched by tetrode and cluster in the file names here. The stable2 file of 1121R157-P11C2 is
15 min long (26,954 samples) and its rate is taken over its own duration.

In [ ]:
dup = []
first = {s: us[0] for s, us in SESS.items()}
for tr in TR:
    keys = {}
    for s, u in first.items():
        x = data[u][tr]
        keys.setdefault((x["d"][:20000].tobytes(), x["x"][:20000].tobytes()), []).append(s)
    for v in keys.values():
        if len(v) > 1:
            a, b = data[first[v[0]]][tr], data[first[v[1]]][tr]
            n = min(len(a["d"]), len(b["d"]))
            dup.append({"trial": tr, "sessions": " and ".join(v), "samples": f"{len(a['d'])} and {len(b['d'])}",
                        "direction identical": np.mean(a["d"][:n] == b["d"][:n]),
                        "x identical": np.mean(a["x"][:n] == b["x"][:n]),
                        "first timestamps": f"{a['first_ts']:.0f} and {b['first_ts']:.0f}",
                        "spike samples identical": np.mean(a["n"][:n] == b["n"][:n]) == 1})
print("Tracking (positions and directions) deposited for two different sessions:")
print(pd.DataFrame(dup).to_string(index=False))
ra_ar_same = [np.mean(data[first["1211R039"]]["RA_ar"]["d"][:54000] == data[first["1212R039"]]["RA_ar"]["d"][:54000])]
print("Arena-frame directions of the same two rotating trials identical in a fraction", round(ra_ar_same[0], 3))
spk_dup = {}
for u in U:
    for tr in ["SA1", "RA_rm", "SA2"]:
        spk_dup.setdefault(data[u][tr]["spike_t"].tobytes(), []).append((u, tr))
print("Spike trains deposited twice:", [v for v in spk_dup.values() if len(v) > 1])
DUP_TRACK = {("1015R029", "SA2"), ("1025R029", "SA2"), ("1211R039", "RA_rm"), ("1212R039", "RA_rm")}

Two pairs of trials from different days share their tracking sample for sample: stable2 of
1015R029 and of 1025R029 (rat R029), and the room-frame tracking of the rotating trial of
1211R039 and 1212R039 (rat R039), whose arena-frame tracking differs. Their spike trains differ,
so one trial of each pair carries another day's tracking. Which one is not known; analyses of
position or direction in these trials are shown with and without the four sessions. No spike
train is deposited twice.

In [ ]:
import io

dats = hd20.dat_files()
z = hd20.open_zip()
zones = {}
for s in SESS:
    p = dats[(dats.session == s) & (dats.trial == "RA") & (dats.frame == "room")].path.iloc[0]
    info, _ = hd20.parse_dat(z.read(p))
    room = [float(v) for v in info["RoomTrackReinforcedSector"].split()]
    arena = [float(v) for v in (info["ArenaTrackReinforcedSector"] or info["RoomTrackReinforcedSector"]).split()]
    zones[s] = {"room": room, "arena": arena, "pix_per_cm": float(info["TrackerResolution_PixPerCM"]),
                "centre": [float(v) for v in info["ArenaCenterXY"].split()], "diam_m": float(info["ArenaDiameter_m"])}
print("Tracker files:", len(dats), "; per trial and frame:", dats.groupby(["trial", "frame"]).size().to_dict())
print("Sessions with all four tracker files:",
      int((dats.groupby("session").size() == 4).sum()), "of", dats.session.nunique(),
      "; sessions of the TSD files without tracker files:", sorted(set(SESS) - set(dats.session)))
zt = pd.DataFrame({s: {"room zone (centre, width, inner r, outer r)": " ".join(f"{v:g}" for v in z_["room"]),
                       "arena zone": " ".join(f"{v:g}" for v in z_["arena"]),
                       "pix/cm": z_["pix_per_cm"], "centre": z_["centre"], "diameter m": z_["diam_m"]}
                   for s, z_ in zones.items()}).T
print(zt.drop(columns=["centre"]).value_counts().to_string())
dat_names = {p_.split("/")[-1] for p_ in dats.path}
listed = {lst: set(pd.read_excel(io.BytesIO(z.read(f"data/DAT/{lst}")), header=None, engine="calamine")[0].astype(str).str.strip())
          for lst in ["Rotating_File_List.xls", "Stable_File_List.xls"]}
for lst, names in listed.items():
    print(f"{lst}: {len(names)} names; in the zip's DAT folder: {len(names & dat_names)}; not in the zip: {sorted(names - dat_names)}")
print("DAT files in neither list:", sorted(dat_names - set().union(*listed.values())))


The tracker headers give an arena of 1.2 m at 2.1333 pixels/cm centred on (127.5, 127.5), so the
arena spans about 256 pixels, as the TSD positions do. The shock zone is a 30 degree sector from
radius 63.75 to 127.5 pixels (50% of the radius) or a 45 degree sector from 51 to 127.5 pixels
(40%), as the STAR Methods describe, centred at 90 degrees in both frames in every session. The
check below finds which direction 90 degrees is in pixel coordinates, from the samples the
tracker flags as inside each zone.

In [ ]:
flag_ang = []
for s in list(SESS)[:44]:
    for fr, bit in [("room", 1), ("arena", 2)]:
        p = dats[(dats.session == s) & (dats.trial == "RA") & (dats.frame == fr)].path.iloc[0]
        _, df = hd20.parse_dat(z.read(p))
        m = ((df.sectors.fillna(0).astype(int) & bit) > 0) & (df.x > 0)
        if m.sum() > 10:
            a = np.angle(np.exp(1j * np.arctan2(-(df.y[m] - 127.5), df.x[m] - 127.5)).mean(), deg=True) % 360
            flag_ang.append({"session": s, "frame": fr, "flagged": int(m.sum()), "angle_y_up": a})
flag_ang = pd.DataFrame(flag_ang)
print("Angle of the samples flagged inside the zone, measured with y pointing up (degrees):")
print(flag_ang.groupby("frame").angle_y_up.describe().round(1).to_string())


def zone_xy(z_):
    """Pixel position of the centre of a sector (centre angle, width, inner, outer radius), the
    angle counted with y up, as the flagged samples show."""
    a = np.deg2rad(z_[0])
    r = (z_[2] + z_[3]) / 2
    return 127.5 + r * np.cos(a), 127.5 - r * np.sin(a)

The samples flagged inside each zone lie at about 90 degrees when the angle is counted with y
pointing up (pixel y grows downwards), so the zone centre is placed at (127.5, 127.5 - r) with r
the zone's mid radius. This is consistent with the headers; it is not a check of the zone's
exact edges.

**Clock.** The spikes are stored inside the tracker samples, so they share the tracking clock by
construction. The check below shows that the assignment carries direction information: the
directionality of the stable1 tuning curves with the spike counts as stored, and with the spike
counts shifted against the tracking by 1 to 120 s.

In [ ]:
shift_rows = []
for sh in [0, 1, 3, 10, 30, 120]:
    rs = []
    for u in U:
        x = data[u]["SA1"]
        k = int(round(sh * x["fs"]))
        y = dict(x)
        y["n"] = np.roll(x["n"], k)
        rs.append(hd20.resultant(hd20.tuning(y)[2])[0])
    shift_rows.append({"shift_s": sh, "median_directionality": np.median(rs), "mean": np.mean(rs)})
print(pd.DataFrame(shift_rows).round(3).to_string(index=False))
off = []
for u in U[:20]:
    x = data[u]["SA1"]
    st = x["first_ts"] + x["spike_sample"] * 1e4 / x["fs"]
    off.append((x["spike_t"] - st) / 10)
off = np.concatenate(off)
print("Spike time minus the nominal time of its sample (first timestamp + index / rate), ms, 20 units:",
      np.percentile(off, [0, 5, 50, 95, 100]).round(1))

Two more problems show in the outputs above. The stable1 file of unit 1024R157-P9C1 holds the
same spikes, sample for sample, as its rotating-trial files (2919 spikes), on different
tracking; its stable1 spikes are very likely the rotating trial's. It is kept in the analyses
as deposited (one unit of 115), and the claims that compare stable1 with other trials are not
sensitive to one unit at the level of the tests. One tracker file of session 0803R027 is named
for rat R024 (`0803SA2R024RoomTrack...`). Spike times run ahead of the nominal sample times
(first timestamp plus index over rate) by up to about 3 s by the end of a trial, so the tracker
samples are not exactly evenly spaced; analyses of direction use the sample each spike is
stored in, and the spike-count correlations use the spike times.

## cell-counts

> The paper analyses 115 dorsal MEC head-direction cells from 13 rats, among 353 MEC cells
> forming 556 cell pairs (Table S1), out of 1594 MEC single units recorded.

Reading (a) counts what the data set holds; reading (b) applies the paper's head-direction
criteria (stable1 mean rate below 10 AP/s, Rayleigh test of the Rayleigh vector of the 10 degree
tuning curve, STAR Methods, "Quantifying Head-direction Signals, Directionality", with n the
spikes behind the curve, and Kuiper's test of the spike directions against the sampled
directions, each at p < 0.05; n = 36 bins and the Rayleigh test of the raw spike directions are
variants). Reading (b) is circular, as the deposited units were already selected as head-direction
cells; it checks that the selection holds on the deposited files. The per-rat
counts are compared with Table S1 of the paper (values typed from the table as the reference).

In [ ]:
TABLE_S1 = {  # Table S1: rat: (head-direction cells, MEC cells, MEC cell pairs)
    "R024": (5, 32, 116), "R027": (3, 8, 15), "R029": (41, 63, 141), "R030": (0, 9, 3), "R035": (17, 43, 73),
    "R039": (6, 16, 31), "R084": (2, 2, 1), "R130": (1, 6, 3), "R157": (25, 68, 50), "R176": (1, 2, 1),
    "R357": (8, 10, 14), "R521": (3, 53, 76), "R523": (1, 7, 1), "R848": (2, 20, 18), "R850": (0, 14, 13)}
sess_n = units.groupby("session").size()
pairs_by_rat = (sess_n * (sess_n - 1) // 2).groupby(units.groupby("session").rat.first()).sum()
cls = []
for u in U:
    x = data[u]["SA1"]
    ok = x["d"] <= 360
    spk_dirs = np.repeat(x["d"][ok].astype(float), x["n"][ok])
    # Rayleigh vector of the 10 degree tuning curve (STAR Methods, Quantifying Head-direction
    # Signals, Directionality); its z statistic with n the spikes behind the curve, and n = 36 bins
    # as a variant
    spikes, _, rate_curve = hd20.tuning(x)
    r, _ = hd20.resultant(rate_curve)
    r_spk, _ = hd20.resultant(np.ones(len(spk_dirs)), np.deg2rad(spk_dirs))
    v, pk = hd20.kuiper_two_sample(spk_dirs, x["d"][ok].astype(float))
    cls.append({"unit": u, "rate_SA1": x["n"].sum() / (len(x["d"]) / x["fs"]), "rayleigh_r": r,
                "rayleigh_p": hd20.rayleigh_p(r, spikes.sum()), "rayleigh_p_36": hd20.rayleigh_p(r, 36),
                "rayleigh_p_spikes": hd20.rayleigh_p(r_spk, len(spk_dirs)), "kuiper_V": v, "kuiper_p": pk})
cls = pd.DataFrame(cls).merge(units[["unit", "rat", "session"]])
cls["passes"] = (cls.rate_SA1 < 10) & (cls.rayleigh_p < 0.05) & (cls.kuiper_p < 0.05)
by_rat = pd.DataFrame({
    "HD cells, Table S1": {r: v[0] for r, v in TABLE_S1.items()},
    "HD units in data set": units.groupby("rat").size(),
    "pass criteria (b)": cls.groupby("rat").passes.sum(),
    "simultaneous HD pairs in data set": pairs_by_rat,
    "MEC cells, Table S1": {r: v[1] for r, v in TABLE_S1.items()},
    "MEC pairs, Table S1": {r: v[2] for r, v in TABLE_S1.items()}}).fillna(0).astype(int)
by_rat.loc["total"] = by_rat.sum()
print(by_rat.to_string())
print("\nRats with head-direction units:", units.rat.nunique())
print("Reading (a): head-direction units", len(U), "(expected 115 +/- 11.5); MEC units in the data set", len(U),
      "(expected 353, the other MEC units are not deposited); simultaneously recorded pairs", int(pairs_by_rat.sum()),
      "(all of them head-direction pairs; expected 556 MEC pairs, and 142 head-direction pairs in Figure 4)")
print("Reading (b): units passing the head-direction criteria", int(cls.passes.sum()), "of", len(cls))
print("Failing units:")
print(cls[~cls.passes].round(4).to_string(index=False))
for name, col in [("n = 36 bins", "rayleigh_p_36"), ("Rayleigh test on the spike directions", "rayleigh_p_spikes")]:
    print(f"Variant, {name}: {int(((cls.rate_SA1 < 10) & (cls[col] < 0.05) & (cls.kuiper_p < 0.05)).sum())} pass")
print("\nThe two R523 units (Table S1 lists one R523 and 41 R029 head-direction cells):")
print(units[units.rat == "R523"][["unit", "file_SA1", "file_RA_rm", "file_SA2"]].to_string(index=False))
print("\nRates in stable1 (AP/s): min %.2f, max %.2f; Rayleigh p max %.2g; Kuiper p max %.2g" %
      (cls.rate_SA1.min(), cls.rate_SA1.max(), cls.rayleigh_p.max(), cls.kuiper_p.max()))
for alpha in [0.01, 0.001]:
    print(f"  at p < {alpha}: {int(((cls.rate_SA1 < 10) & (cls.rayleigh_p < alpha) & (cls.kuiper_p < alpha)).sum())} pass")

**Result: partially reproduced.** The data set holds 115 head-direction units from 13 rats,
matching the paper's count; their per-rat numbers match Table S1 except R029 (40 against 41)
and R523 (2 against 1). Both R523 units are named R523 in all their files (session 1001R523), so
the files do not show which unit the table counts under another rat. Applying the paper's
criteria at p < 0.05, with the Rayleigh test of the tuning curve's vector, 110 of the 115 pass:
two units fire above 10 AP/s in stable1 (13.8 and 11.9 AP/s), two have a Kuiper p of 0.051 and
0.054, and three have a Rayleigh p of 0.28 to 0.43 (two of them among the Kuiper failures). With
n = 36 bins in the Rayleigh test 83 pass, and with the Rayleigh test of the raw spike directions
110. The 115 are within tolerance under reading (a) and 110 under reading (b), which is circular
since the deposited units were selected as head-direction cells. The 353 MEC cells and 556 MEC
cell pairs of Table S1 cannot be counted: only the head-direction cells are deposited. The units
form 146 simultaneously recorded pairs, against the 142 head-direction pairs the paper analyses
in Figure 4.

## hd-cell-rate

> The head-direction cells have long-duration spikes and a mean firing rate of 1.41 ± 0.20 AP/s.

Mean over units of spike count over trial duration, (a) stable1, (b) the mean over the three
trials. Spike width cannot be tested (no waveforms).

In [ ]:
rate = pd.DataFrame({tr: {u: data[u][tr]["n"].sum() / TRIALDUR[(u, tr)] for u in U} for tr in ["SA1", "RA_rm", "SA2"]})
rate.columns = ["stable1", "rotating", "stable2"]
ra = rate.stable1.mean()
rb = rate.mean(axis=1).mean()
print(f"(a) stable1: mean {ra:.3f} AP/s, SEM {rate.stable1.sem():.3f}, median {rate.stable1.median():.3f}, "
      f"range {rate.stable1.min():.3f} to {rate.stable1.max():.3f}")
print(f"(b) mean over three trials: {rb:.3f} AP/s, SEM {rate.mean(axis=1).sem():.3f}")
print("Expected 1.41 +/- 0.20 AP/s; within under (a):", abs(ra - 1.41) <= 0.2, "; under (b):", abs(rb - 1.41) <= 0.2)
print("Units at or above 10 AP/s in stable1:", int((rate.stable1 >= 10).sum()))
print("\nBy rat (stable1):")
print(rate.join(units.set_index("unit").rat).groupby("rat").stable1.agg(["size", "mean", "median"]).round(2).T.to_string())

**Result: reproduced.** The mean stable1 rate is 1.392 AP/s (reading (a)) and the mean over the
three trials 1.433 AP/s (reading (b)), both within 1.41 ± 0.20 AP/s. The median is 0.668 AP/s,
and two units fire above 10 AP/s in stable1, the paper's upper limit for these cells.

## fig1a-avoidance-learning

Blocked, data missing. The learning curves are over the pretraining and training sessions. The
data set's tracker files hold only the 44 recording sessions (stable1, rotating, stable2), one
triad per session, with 1 to 12 sessions per rat; the training sessions are not deposited.

## fig1c-speed

> Rotation did not change locomotor speed relative to stable1, measured in either frame, but
> speed was lower in stable2 than in the other conditions.

Mean speed per session in stable1, rotating room frame, rotating arena frame and stable2 from the
30 Hz TSD positions (pixels, 2.1333 pixels/cm from the tracker headers), samples with x = y = 0
left out. Readings: (a) displacement between successive tracked samples, (b) displacement over 1
s steps (30 samples). Repeated-measures ANOVA on sessions, Greenhouse-Geisser when Mauchly's
test is significant, and Tukey HSD. The paper ran its tests in JMP and names Tukey-Kramer HSD
after repeated-measures ANOVAs; the post hoc test here uses the repeated-measures error term of
the ANOVA (`hd20.tukey_rm`, an assumption), and the between-groups Tukey-Kramer HSD, which ignores
the pairing, is a variant. The four sessions with duplicated tracking are left out of the main
test and kept in a variant.

In [ ]:
def speed(x, step):
    ok = ~((x["x"] == 0) & (x["y"] == 0))
    px, py = x["x"].astype(float), x["y"].astype(float)
    px[~ok] = np.nan
    py[~ok] = np.nan
    dist = np.hypot(px[step:] - px[:-step], py[step:] - py[:-step])
    return np.nanmean(dist) / 2.1333 * x["fs"] / step   # cm/s


SPEED_COND = {"SA1": "stable1", "RA_rm": "rotating room", "RA_ar": "rotating arena", "SA2": "stable2"}
dup_sessions = {s for s, _ in DUP_TRACK}
for label, step in [("(a) successive samples", 1), ("(b) 1 s steps", 30)]:
    sp = pd.DataFrame({SPEED_COND[tr]: {s: speed(data[us[0]][tr], step) for s, us in SESS.items()} for tr in TR})
    for name, w in [("main, duplicated-tracking sessions left out", sp.drop(index=sorted(dup_sessions))),
                    ("variant, all sessions", sp)]:
        r = hd20.rm_anova(w)
        print(f"{label}, {name}: means (cm/s)", w.mean().round(2).to_dict())
        print("  ANOVA", hd20.fmt_anova(r))
        print("  Tukey HSD with the repeated-measures error term (main):")
        print(hd20.tukey_rm(w).round(4).to_string(index=False))
        print("  Tukey-Kramer HSD between groups (variant):")
        print(hd20.tukey(w).round(4).to_string(index=False))
        pt = []
        for c in ["stable1", "rotating room", "rotating arena"]:
            t = stats.ttest_rel(w[c], w["stable2"])
            pt.append(f"{c} vs stable2 paired t p = {t.pvalue:.2g}, stable2 lower in {int((w['stable2'] < w[c]).sum())}/{len(w)}")
        print("  " + "; ".join(pt))
    if step == 1:
        sp_a = sp
print("\nSpeed range over sessions and conditions, reading (a): %.2f to %.2f cm/s" % (sp_a.min().min(), sp_a.max().max()))
fig, ax = plt.subplots(figsize=(5, 3))
ax.boxplot([sp_a[c] for c in sp_a.columns], tick_labels=list(sp_a.columns))
ax.set_ylabel("speed (cm/s), reading (a)")
plt.show()

**Result: reproduced.** With the 40
sessions whose tracking is not duplicated, speed differs across conditions (reading (a): F(1.94,
75.59) = 64.80, p = 7.6e-17; reading (b): F(1.87, 72.82) = 55.86, p = 6.9e-15). Under reading (a)
stable2 is lower than each other condition by Tukey HSD with the repeated-measures error term
(adjusted p below 0.0001; 10.76 against 14.63 to 15.60 cm/s), and stable1 and the two rotating
frames do not differ (adjusted p 0.08 to 1.00). Under reading (b) the room and arena frames
differ (adjusted p = 0.0045), so only reading (a) gives the claimed pattern. With the
between-groups Tukey-Kramer HSD (variant) both readings give it (others adjusted p 0.07 to 1.00).
Reading (a) gives the pattern under either error term, so the outcome does not rest on that
choice. With all 44 sessions, reading (a) gives a room-arena difference at adjusted p = 0.048
under the repeated-measures error term, and the pattern under the between-groups variant.

## fig1c-lfp-power

Blocked, data missing: the data set holds no LFP.

## fig1c-firing-rate

> Firing rates of the head-direction cells were maintained across stable1, rotating and stable2.

Mean rate per unit and trial; repeated-measures ANOVA over the three trials, Greenhouse-Geisser
when Mauchly's test is significant. A claim of absence: reproduced when p ≥ 0.05.

In [ ]:
r = hd20.rm_anova(rate)
print("Mean rate (AP/s):", rate.mean().round(3).to_dict(), "; median:", rate.median().round(3).to_dict())
print("ANOVA", hd20.fmt_anova(r), "; uncorrected p =", round(r["p_uncorrected"], 4))
print("Mean change rotating - stable1: %.3f AP/s; stable2 - stable1: %.3f AP/s" %
      ((rate.rotating - rate.stable1).mean(), (rate.stable2 - rate.stable1).mean()))
print("Wilcoxon signed-rank (variant): rotating vs stable1 p = %.3g; stable2 vs stable1 p = %.3g" %
      (stats.wilcoxon(rate.rotating, rate.stable1).pvalue, stats.wilcoxon(rate.stable2, rate.stable1).pvalue))
print("Without unit 1024R157-P9C1:", hd20.fmt_anova(hd20.rm_anova(rate.drop(index="1024R157-P9C1"))))
hi = rate.rotating.idxmax()
print(f"Highest rotating rate: {hi}, {rate.loc[hi].round(2).to_dict()} AP/s; without it:",
      hd20.fmt_anova(hd20.rm_anova(rate.drop(index=hi))), "; without it and 1024R157-P9C1:",
      hd20.fmt_anova(hd20.rm_anova(rate.drop(index=[hi, "1024R157-P9C1"]))))
print("Log rates (variant):", hd20.fmt_anova(hd20.rm_anova(np.log(rate + 0.01))))
print("\nBy rat, mean rate per trial:")
print(rate.join(units.set_index("unit").rat).groupby("rat").mean().round(2).T.to_string())

**Result: reproduced.** Mean rates are 1.392, 1.710 and 1.198 AP/s in stable1, rotating and
stable2; the repeated-measures ANOVA with Greenhouse-Geisser correction gives F(1.35, 154.10) =
1.76, p = 0.184 (the paper: F(1.34, 153.21) = 1.89, p = 0.17). The mean change from stable1 is
+0.318 AP/s in rotation and -0.194 AP/s in stable2. Leaving out the unit with the highest
rotating rate (1001R523-P9C1, 40.1 AP/s in rotation against 7.8 in stable1) gives p = 0.275, and
leaving out it and 1024R157-P9C1 p = 0.29, so the absence does not rest on that unit. A Wilcoxon
test (variant) finds rotating different from stable1 (p = 0.02), and on log rates the ANOVA is
significant (p = 0.0015), so the absence rests on the paper's test on raw rates.

## fig1c-directionality

> Directional tuning of head-direction cells was strongly degraded during rotation in both
> frames, more in the arena frame than in the room frame, and returned in stable2.

Directionality is the Rayleigh vector length of the 10 degree tuning curve, per unit in four
conditions. Repeated-measures ANOVA and Tukey HSD with the ANOVA's repeated-measures error term
(the paper's F(3, 342) on 115 cells implies a within-cell error term; that the post hoc test used
it is an assumption). The between-groups Tukey-Kramer HSD and Bonferroni-corrected paired t-tests
are variants.

In [ ]:
curves = {u: {tr: hd20.tuning(data[u][tr]) for tr in TR} for u in U}
dirn = pd.DataFrame({hd20.TRIAL_NAMES[tr]: {u: hd20.resultant(curves[u][tr][2])[0] for u in U} for tr in TR})
r = hd20.rm_anova(dirn)
print("Directionality mean:", dirn.mean().round(3).to_dict())
print("median:", dirn.median().round(3).to_dict())
print("ANOVA", hd20.fmt_anova(r))
print("Tukey HSD with the repeated-measures error term (main):")
print(hd20.tukey_rm(dirn).round(5).to_string(index=False))
print("Tukey-Kramer HSD between groups (variant):")
print(hd20.tukey(dirn).round(5).to_string(index=False))
pt = {f"{a} vs {b}": stats.ttest_rel(dirn[a], dirn[b]).pvalue for a, b in
      [("stable1", "stable2"), ("stable1", "rotating room"), ("stable2", "rotating room"), ("rotating room", "rotating arena"),
       ("stable1", "rotating arena"), ("stable2", "rotating arena")]}
print("Paired t-tests (variant), and Bonferroni-corrected over the 6 pairs:",
      {k: f"{v:.2g} / {min(6 * v, 1):.2g}" for k, v in pt.items()})
print("Units with room > arena directionality: %d of %d" % ((dirn["rotating room"] > dirn["rotating arena"]).sum(), len(dirn)))
print("\nBy rat, mean directionality:")
print(dirn.join(units.set_index("unit").rat).groupby("rat").mean().round(3).T.to_string())
print("Range of directionality: %.3f to %.3f" % (dirn.min().min(), dirn.max().max()))
fig, ax = plt.subplots(figsize=(6, 3))
ax.boxplot([dirn[c] for c in dirn.columns], tick_labels=list(dirn.columns))
ax.set_ylabel("Rayleigh vector length")
plt.show()

**Result: reproduced, resting on the assumed error term of the post hoc test.** Directionality
differs across conditions (F(2.77, 315.60) = 72.66, p = 8.9e-34; the paper F(3, 342) = 70.36).
With Tukey HSD on the repeated-measures error term, stable1 and stable2 do not differ (p = 0.27),
the room frame is below stable1 (p below 0.0001) and stable2 (p = 0.0019), and the arena frame is
below the room frame (p below 0.0001), every required pair at p < 0.01. With the between-groups
Tukey-Kramer HSD (variant), which ignores the pairing of the conditions within a cell, the room
frame is not below stable2 at p < 0.01 (p = 0.055), so that variant gives the partial outcome.
Bonferroni-corrected paired t-tests agree with the main reading (stable2 against room p = 0.0048,
stable1 against stable2 p = 0.14).

## fig1d-preference-change and fig1d-instability

> The change in preferred direction differs across trial pairs (stable1-stable2 =
> stable1-rotating room frame > stable1-rotating arena frame), and is greater for every trial
> pair than between the two halves of stable1.

> Directional instability (the KLD between tuning curves) differs across trial pairs and is
> greater for every trial pair than between the two halves of stable1.

Preferred direction: the bin of maximum rate; change: absolute circular difference. KLD of the
Krichevsky-Trofimov probabilities, readings (a) spike counts per bin, (b) rates per bin. Halves
of stable1: first and second 15 min (claims file). Repeated-measures ANOVA over the three trial
pairs with Tukey HSD (repeated-measures error term; the between-groups Tukey-Kramer HSD as a
variant); one-way ANOVA of the four groups with Dunnett's test against the halves. A variant of
KLD (b) rescales each rate curve to sum to the cell's spike count before the
Krichevsky-Trofimov estimate, since adding 0.5 to rates far below 1 makes the probabilities
nearly uniform.

In [ ]:
PAIRS = {"s1-s2": ("SA1", "SA2"), "s1-room": ("SA1", "RA_rm"), "s1-arena": ("SA1", "RA_ar")}
halves = {}
for u in U:
    n = len(data[u]["SA1"]["d"])
    halves[u] = (hd20.tuning(data[u]["SA1"], slice(0, n // 2)), hd20.tuning(data[u]["SA1"], slice(n // 2, n)))
pref, kl_c, kl_r, kl_s = {}, {}, {}, {}
for u in U:
    c = curves[u]
    pref[u] = {k: float(hd20.circ_absdiff(hd20.preferred(c[a][2]), hd20.preferred(c[b][2]))) for k, (a, b) in PAIRS.items()}
    pref[u]["halves"] = float(hd20.circ_absdiff(hd20.preferred(halves[u][0][2]), hd20.preferred(halves[u][1][2])))
    kl_c[u] = {k: hd20.kld(c[a][0], c[b][0]) for k, (a, b) in PAIRS.items()}
    kl_c[u]["halves"] = hd20.kld(halves[u][0][0], halves[u][1][0])
    kl_r[u] = {k: hd20.kld(c[a][2], c[b][2]) for k, (a, b) in PAIRS.items()}
    kl_r[u]["halves"] = hd20.kld(halves[u][0][2], halves[u][1][2])
    # variant: rates rescaled to sum to the spike count, so that the Krichevsky-Trofimov 0.5 acts on
    # the scale of counts (low rates otherwise put most of the probability in the 0.5 terms)
    sc = lambda cv: cv[2] * cv[0].sum() / cv[2].sum() if cv[2].sum() > 0 else cv[2]
    kl_s[u] = {k: hd20.kld(sc(c[a]), sc(c[b])) for k, (a, b) in PAIRS.items()}
    kl_s[u]["halves"] = hd20.kld(sc(halves[u][0]), sc(halves[u][1]))
pref, kl_c, kl_r, kl_s = pd.DataFrame(pref).T, pd.DataFrame(kl_c).T, pd.DataFrame(kl_r).T, pd.DataFrame(kl_s).T


def fig1d_tests(w, name):
    tri = w[["s1-s2", "s1-room", "s1-arena"]]
    r1 = hd20.rm_anova(tri)
    r2 = hd20.oneway({c: w[c] for c in w.columns})
    dn = hd20.dunnett({c: w[c] for c in w.columns}, "halves")
    print(f"{name}: means", w.mean().round(4).to_dict(), "; medians", w.median().round(4).to_dict())
    print("  trial pairs, RM ANOVA", hd20.fmt_anova(r1))
    print("  Tukey HSD with the repeated-measures error term (main):")
    print(hd20.tukey_rm(tri).round(5).to_string(index=False))
    print("  Tukey-Kramer HSD between groups (variant):")
    print(hd20.tukey(tri).round(5).to_string(index=False))
    print(f"  four groups, one-way ANOVA F({r2['df1']}, {r2['df2']}) = {r2['F']:.2f}, p = {r2['p']:.3g}")
    print(dn.round(5).to_string(index=False))
    return r1, r2, dn


res_pref = fig1d_tests(pref, "Preferred direction change (degrees)")
print()
res_klc = fig1d_tests(kl_c, "KLD, reading (a) spike counts")
print()
res_klr = fig1d_tests(kl_r, "KLD, reading (b) rates")
print()
res_kls = fig1d_tests(kl_s, "KLD, variant: rates rescaled to the spike count")
print("\nBy rat, mean preferred-direction change and KLD (a):")
print(pref.join(units.set_index("unit").rat).groupby("rat").mean().round(1).T.to_string())
print(kl_c.join(units.set_index("unit").rat).groupby("rat").mean().round(3).T.to_string())
print("Ranges: preference change %.0f to %.0f degrees; KLD (a) %.4f to %.4f; KLD (b) %.4f to %.4f" %
      (pref.min().min(), pref.max().max(), kl_c.min().min(), kl_c.max().max(), kl_r.min().min(), kl_r.max().max()))

**Result: partially reproduced (fig1d-preference-change).** The change in preferred direction
differs across trial pairs (F(2, 228) = 17.80, p = 6.6e-08; the paper F(2, 228) = 10.54) and in
the four-group ANOVA (F(3, 456) = 19.62, p = 5.6e-12), and every trial pair exceeds the stable1
halves by Dunnett's test (p = 0.019, 0.0073 and below 0.0001). Stable1-stable2 and stable1-room
do not differ (Tukey p = 0.94 on the repeated-measures error term, 0.95 between groups), as in
the legend, but the stable1-arena change is larger than the other two (91 against 58 and 60
degrees, p below 0.0001), where the legend orders it below them. Whether the legend's order is
garbled is asked of a reader (questions/park-2019.yaml).

**Result: partially reproduced (fig1d-instability).** With KLD on spike counts (reading (a))
both ANOVAs are significant (trial pairs F(1.60, 182.42) = 11.74, p = 7.2e-05; the paper F(1.62,
184.22) = 7.78; four groups F(3, 456) = 13.74, p = 1.4e-08), and stable1-stable2 and
stable1-arena exceed the halves by Dunnett's test (p = 0.0044 and below 0.0001), but stable1-room
does not (p = 0.13). With KLD on rates (reading (b)) only stable1-arena exceeds the halves (p =
0.0008). With the rates rescaled to the spike count before the Krichevsky-Trofimov estimate
(variant) the same holds (stable1-arena p below 0.0001, the others 0.15 and 0.26).

## fig2-decoding-stable-vs-rotating

> Head direction decoded from stable1 ensemble tuning is relatively accurate in stable2 and
> relatively inaccurate and abruptly changing in both frames during rotation, although for some
> ensembles room-frame decoding is similar to stable2; seven ensembles of at least five
> head-direction cells were analysed.

The decoder of HD_Max_likelihood.py (helper `fit_decoder`, `decode`): trained on stable1, applied
to stable2 and to the rotating trial in room and arena coordinates; the estimate is the start
of the winning 10 degree bin, as the code's `angs`. Windows of 20 samples (660 ms; main) and 4
samples (133 ms; variant). Error measures (a) median and (b) mean absolute circular error per
ensemble and condition.

In [ ]:
ENS = {s: us for s, us in SESS.items() if len(us) >= 5}
print("Sessions with at least 5 head-direction units:", len(ENS), {s: len(us) for s, us in ENS.items()})
DEC = {}
for w in [20, 4]:
    for s, cells in ENS.items():
        dtr, ctr = hd20.ensemble_windows(data, cells, "SA1", w)
        lam = hd20.fit_decoder(dtr, ctr)
        for tr in ["SA2", "RA_rm", "RA_ar"]:
            dte, cte = hd20.ensemble_windows(data, cells, tr, w)
            est = hd20.decode(lam, cte)
            DEC[(w, s, tr)] = {"dir": dte, "est": est, "err": hd20.signed_error(est, dte)}

In [ ]:
rows = []
for (w, s, tr), v in DEC.items():
    ok = ~np.isnan(v["err"])
    e = np.abs(v["err"][ok])
    rows.append({"window": w, "ensemble": s, "cells": len(ENS[s]), "trial": tr, "decoded": int(ok.sum()),
                 "windows": len(v["err"]), "median_abs_err": np.median(e), "mean_abs_err": e.mean(),
                 "frac_within_30": np.mean(e <= 30), "change_gt_90": np.mean(np.abs((np.diff(v["est"][ok]) + 180) % 360 - 180) > 90)})
dec = pd.DataFrame(rows)
print("Chance level: absolute error uniform on 0-180 degrees has median and mean 90.")
for w in [20, 4]:
    print(f"\nWindow of {w} samples ({w / 30.02 * 1000:.0f} ms):")
    t = dec[dec.window == w].pivot_table(index=["ensemble", "cells"], columns="trial",
                                         values=["median_abs_err", "mean_abs_err", "decoded"])
    print(t.round(1).to_string())
    for m in ["median_abs_err", "mean_abs_err"]:
        p = dec[dec.window == w].pivot(index="ensemble", columns="trial", values=m)
        print(f"  {m}: stable2 below arena in {int((p.SA2 < p.RA_ar).sum())} of {len(p)}, "
              f"below room in {int((p.SA2 < p.RA_rm).sum())} of {len(p)}; "
              f"room below arena in {int((p.RA_rm < p.RA_ar).sum())} of {len(p)}")
ref = []
for s, cells in ENS.items():
    dtr, ctr = hd20.ensemble_windows(data, cells, "SA1", 20)
    h = len(dtr) // 2
    e = np.abs(hd20.signed_error(hd20.decode(hd20.fit_decoder(dtr[:h], ctr[:, :h]), ctr[:, h:]), dtr[h:]))
    ref.append({"ensemble": s, "first half -> second half of stable1, median abs err": np.nanmedian(e),
                "mean": np.nanmean(e)})
print("\nReference: decoding the second half of stable1 from its first half (20 samples):")
print(pd.DataFrame(ref).round(1).to_string(index=False))
print("\nFraction of successive decoded windows whose estimate jumps by more than 90 degrees (20 samples):")
print(dec[dec.window == 20].pivot(index="ensemble", columns="trial", values="change_gt_90").round(3).to_string())

**Result: reproduced, resting on the claims file's assumption of a majority.** Seven sessions
hold at least five head-direction units. With 20-sample windows the stable2 error is below the
arena-frame error in 4 of 7 ensembles and below the room-frame error in 4 of 7, by the median
and by the mean absolute error, and the same counts hold with 4-sample windows. Stable2 decoding
is close to or worse than chance (90 degrees) in 1026R157, 1117R157 and 1121R157 (median 102 to
135 degrees), where decoding the second half of stable1 from its first half is also poor (65 to
116 degrees), so these ensembles carry little decodable direction. The room frame is decoded
better than the arena frame in 6 of 7.

## figs2-frame-preference

> During rotation, frame-specific directional discharge alternated between signaling room and
> arena direction; four of seven ensembles showed an overall room preference and two of seven an
> arena preference.

Idir per cell and 133 ms interval (4 samples) of the rotating trial, in room and in arena
directions; ΔIdir the room sum over cells minus the arena sum. Readings named by the criterion:
direction bins of 10 or 45 degrees; probabilities from the rotating trial itself or from
stable1; the SD of the 2 SD rule from the ensemble's ΔIdir series or from its stable1 Idir sum;
the no-preference margin, the two probabilities within a factor of 2, or both below 0.05.

In [ ]:
IDIR = {}
for bins in [10, 45]:
    for src in ["rotating", "stable1"]:
        for s, cells in ENS.items():
            n_ra = min(len(data[c]["RA_rm"]["d"]) for c in cells)
            d_rm, c_ra = hd20.ensemble_windows(data, cells, "RA_rm", 4)
            d_ar, _ = hd20.ensemble_windows(data, cells, "RA_ar", 4)
            d_s1, c_s1 = hd20.ensemble_windows(data, cells, "SA1", 4)
            room = np.zeros(len(d_rm))
            arena = np.zeros(len(d_rm))
            s1 = np.zeros(len(d_s1))
            for i in range(len(cells)):
                if src == "rotating":
                    room += hd20.idir(c_ra[i], d_rm, c_ra[i], d_rm, bins)
                    arena += hd20.idir(c_ra[i], d_ar, c_ra[i], d_ar, bins)
                else:
                    room += hd20.idir(c_ra[i], d_rm, c_s1[i], d_s1, bins)
                    arena += hd20.idir(c_ra[i], d_ar, c_s1[i], d_s1, bins)
                s1 += hd20.idir(c_s1[i], d_s1, c_s1[i], d_s1, bins)
            IDIR[(bins, src, s)] = {"room": room, "arena": arena, "delta": room - arena, "s1": s1,
                                    "d_rm": d_rm, "d_ar": d_ar}

In [ ]:
pref_rows = []
for (bins, src, s), v in IDIR.items():
    dl = v["delta"][~np.isnan(v["delta"])]
    for sd_name, sd in [("delta series", np.std(dl)), ("stable1 Idir", np.nanstd(v["s1"]))]:
        p_room, p_arena = np.mean(dl > 2 * sd), np.mean(dl < -2 * sd)
        pref_rows.append({"bins": bins, "probabilities": src, "sd": sd_name, "ensemble": s,
                          "intervals": len(dl), "p_room": p_room, "p_arena": p_arena})
pr = pd.DataFrame(pref_rows)


def label(r, margin):
    hi, lo = max(r.p_room, r.p_arena), min(r.p_room, r.p_arena)
    if margin == "factor 2" and (lo > 0 and hi / lo < 2 or hi == 0):
        return "none"
    if margin == "both < 0.05" and hi < 0.05:
        return "none"
    return "room" if r.p_room > r.p_arena else ("arena" if r.p_arena > r.p_room else "none")


summ = []
for margin in ["factor 2", "both < 0.05"]:
    pr[margin] = pr.apply(lambda r: label(r, margin), axis=1)
    for key, g in pr.groupby(["bins", "probabilities", "sd"]):
        summ.append({"bins": key[0], "probabilities": key[1], "sd": key[2], "margin": margin,
                     "room": int((g[margin] == "room").sum()), "arena": int((g[margin] == "arena").sum()),
                     "none": int((g[margin] == "none").sum())})
summ = pd.DataFrame(summ)
summ["matches 4 and 2"] = (summ.room == 4) & (summ.arena == 2)
summ["room > arena"] = summ.room > summ.arena
print(summ.to_string(index=False))
print("\nReadings giving 4 room and 2 arena:", int(summ["matches 4 and 2"].sum()), "of", len(summ),
      "; readings with more room- than arena-preferring ensembles:", int(summ["room > arena"].sum()))
for b_ in [10, 45]:
    print(f"\nPer ensemble, {b_} degree bins, rotating-trial probabilities:")
    print(pr[(pr.bins == b_) & (pr.probabilities == "rotating")].round(3).to_string(index=False))

**Result: reproduced under one reading of sixteen.** The counts of 4 room- and 2
arena-preferring ensembles are found with 45 degree bins, probabilities from the rotating trial,
the SD of the ΔIdir series and the "both below 0.05" margin. Under that reading the
room-preferring ensembles are 0930R029, 1004R029, 1022R029 and 1117R157 (p above 2 SD 0.605,
0.291, 0.199 and 0.054 against 0.000, 0.000, 0.000 and 0.030 below) and the arena-preferring
1023R029 (0.000 against 0.367) and 1121R157 (0.041 against 0.061); two of the six labels rest on
differences of 0.02 to 0.03. The other 15 readings give 0 to 5 room-preferring and 0 to 1
arena-preferring ensembles; 14 of 16 give more room- than arena-preferring ensembles, which is
the claim's partial form. The outcome rests on the choice of reading, which the paper does not
fix.

## fig2c-proximity-anticorrelation

> The rat's relative proximity to one shock zone versus the other is anti-correlated with
> frame-specific directional information ... (mean r = -0.141 ± 0.029, all p < 0.0001) and for
> ΔIdir (r = -0.18, p < 0.05, in the exemplar).

Relative proximity at each 133 ms interval from the mean tracked position in room coordinates
(distance to the room zone) and in arena coordinates (distance to the arena zone); zone geometry
from the tracker headers (data check). Readings (a) (d_arena - d_room) / (d_arena + d_room), (b)
d_arena - d_room, positive when nearer the room zone. Pearson r of the room Idir sum with
proximity and of the arena Idir sum with minus proximity (14 values), and of ΔIdir with
proximity; under each Idir reading of figs2. The paper does not say how the distance to a zone
is measured, so three are run: to the zone's centre, to its nearest point (0 inside the zone),
and the angle about the arena centre between the rat and the zone's centre direction.

In [ ]:
def interval_xy(x, n, w=4):
    nw = n // w
    px = x["x"][: nw * w].astype(float).reshape(nw, w)
    py = x["y"][: nw * w].astype(float).reshape(nw, w)
    ok = ~((px == 0) & (py == 0))
    with np.errstate(invalid="ignore"):
        return np.where(ok, px, 0).sum(1) / ok.sum(1), np.where(ok, py, 0).sum(1) / ok.sum(1)


def zone_distance(x, y, z_, kind):
    """Distance (pixels, or degrees for 'angle') from positions to a sector zone (centre angle, width,
    inner, outer radius; angles counted with y up): to its centre, to its nearest point (0 inside),
    or the angle about the arena centre between the position and the zone's centre direction."""
    dx, dy = x - 127.5, -(y - 127.5)
    if kind == "centre":
        cx, cy = zone_xy(z_)
        return np.hypot(x - cx, y - cy)
    th = np.rad2deg(np.arctan2(dy, dx)) % 360
    if kind == "angle":
        return hd20.circ_absdiff(th, z_[0])
    r = np.hypot(dx, dy)
    inside = hd20.circ_absdiff(th, z_[0]) <= z_[1] / 2
    d_in = np.maximum.reduce([z_[2] - r, r - z_[3], np.zeros_like(r)])
    d_edge = []
    for phi in np.deg2rad([z_[0] - z_[1] / 2, z_[0] + z_[1] / 2]):
        t = np.clip(dx * np.cos(phi) + dy * np.sin(phi), z_[2], z_[3])   # nearest point on the edge segment
        d_edge.append(np.hypot(dx - t * np.cos(phi), dy - t * np.sin(phi)))
    return np.where(inside, d_in, np.minimum(*d_edge))


PROX = {}
for s, cells in ENS.items():
    n = min(len(data[c]["RA_rm"]["d"]) for c in cells)
    rx, ry = interval_xy(data[cells[0]]["RA_rm"], n)
    ax_, ay_ = interval_xy(data[cells[0]]["RA_ar"], n)
    for kind in ["centre", "edge", "angle"]:
        d_room = zone_distance(rx, ry, zones[s]["room"], kind)
        d_arena = zone_distance(ax_, ay_, zones[s]["arena"], kind)
        with np.errstate(invalid="ignore", divide="ignore"):
            PROX[(s, kind, "a")] = np.where(d_arena + d_room > 0, (d_arena - d_room) / (d_arena + d_room), 0.0)
        PROX[(s, kind, "b")] = d_arena - d_room
rows = []
for (bins, src, s), v in IDIR.items():
    for kind in ["centre", "edge", "angle"]:
        for pr_name in ["a", "b"]:
            prox = PROX[(s, kind, pr_name)]
            for name, y, sign in [("room Idir", v["room"], 1), ("arena Idir", v["arena"], -1), ("delta Idir", v["delta"], 1)]:
                ok = ~np.isnan(y) & ~np.isnan(prox)
                r_, p_ = stats.pearsonr(y[ok], sign * prox[ok])
                rows.append({"bins": bins, "probabilities": src, "distance": kind, "proximity": pr_name, "ensemble": s,
                             "series": name, "r": r_, "p": p_, "n": int(ok.sum())})
cor = pd.DataFrame(rows)
KEYS = ["bins", "probabilities", "distance", "proximity"]
fs_ = cor[cor.series != "delta Idir"].assign(neg_sig=lambda x: (x.r < 0) & (x.p < 1e-4))
g = fs_.groupby(KEYS, sort=False).agg(mean_r=("r", "mean"), sem_r=("r", "sem"), n_neg=("r", lambda r: int((r < 0).sum())),
                                      n=("r", "size"), n_neg_sig=("neg_sig", "sum")).reset_index()
dg = cor[cor.series == "delta Idir"].assign(neg05=lambda x: (x.r < 0) & (x.p < 0.05)).groupby(KEYS, sort=False).agg(
    delta_mean_r=("r", "mean"), delta_n_neg_p05=("neg05", "sum")).reset_index()
g = g.merge(dg)
g["mean within -0.141 +/- 0.029"] = (g.mean_r - -0.141).abs() <= 0.029
g["mean within -0.132 +/- 0.024"] = (g.mean_r - -0.132).abs() <= 0.024
g["all 14 negative, p < 1e-4"] = g.n_neg_sig == 14
print("Frame-specific Idir against relative proximity, 7 ensembles x 2 frames; distance to the zone's centre,")
print("its nearest point (edge) or the angle about the arena centre to the zone's centre direction:")
print(g.round(4).to_string(index=False))
print("\nReadings with the mean within either tolerance:", int((g["mean within -0.141 +/- 0.029"] | g["mean within -0.132 +/- 0.024"]).sum()),
      "of", len(g), "; with all 14 negative at p < 1e-4:", int(g["all 14 negative, p < 1e-4"].sum()),
      "; mean r from %.3f to %.3f" % (g.mean_r.min(), g.mean_r.max()))
print("\nPer ensemble, 10 degree bins, rotating-trial probabilities, proximity (a), each distance:")
print(cor[(cor.bins == 10) & (cor.probabilities == "rotating") & (cor.proximity == "a")]
      .pivot(index="ensemble", columns=["distance", "series"], values="r").round(3).to_string())
print("Proximity (a) range, distance to centre: %.2f to %.2f" % (
    np.nanmin(np.concatenate([v for k, v in PROX.items() if k[1:] == ("centre", "a")])),
    np.nanmax(np.concatenate([v for k, v in PROX.items() if k[1:] == ("centre", "a")]))))

**Result: partially reproduced.** Under every reading (Idir bins and probabilities, proximity
(a) or (b), and distance to the zone's centre, its nearest point or its direction) the mean
correlation of frame-specific Idir with relative proximity is negative (-0.042 to -0.087 over 14
values), but none of the 24 is within -0.141 ± 0.029 or -0.132 ± 0.024; 8 to 12 of the 14
correlations are negative and 7 to 11 are negative with p < 0.0001, never all 14. ΔIdir
correlates negatively with proximity in 4 to 6 of 7 ensembles at p < 0.05 (mean r -0.040 to
-0.128). In the per-ensemble table (10 degree bins, rotating-trial probabilities, proximity (a)),
1026R157 and 1117R157 give positive or near-zero correlations (-0.062 to 0.113) under every
distance. The way the distance is measured does not change the outcome.

## figs3-no-head-direction-bias

> The frame preference ΔIdir does not vary systematically with head direction; its polar
> distribution over head direction in either frame is not biased toward the shock zones and does
> not differ from uniform.

Per ensemble and frame, the mean ΔIdir in 10 degree head-direction bins, normalised to 0 to 1
(adding the magnitude of the minimum and dividing by the maximum, Figure S3 legend), and a
Rayleigh test with the normalised values as weights; n is (a) the number of bins or (b) the sum
of the weights. A claim of absence: reproduced when every test has p ≥ 0.05. Run under each
Idir reading of figs2.

In [ ]:
rows = []
for (bins, src, s), v in IDIR.items():
    for frame, dirs in [("room", v["d_rm"]), ("arena", v["d_ar"])]:
        ok = ~np.isnan(v["delta"]) & ~np.isnan(dirs)
        b = (dirs[ok] // 10).astype(int) % 36
        m = np.bincount(b, weights=v["delta"][ok], minlength=36) / np.maximum(np.bincount(b, minlength=36), 1)
        wts = m + abs(m.min())
        wts = wts / wts.max()
        r_, _ = hd20.resultant(wts)
        rows.append({"bins": bins, "probabilities": src, "ensemble": s, "frame": frame, "r": r_,
                     "p_n_bins": hd20.rayleigh_p(r_, 36), "p_n_weights": hd20.rayleigh_p(r_, wts.sum())})
s3 = pd.DataFrame(rows)
print(s3.groupby(["bins", "probabilities"]).agg(
    mean_r=("r", "mean"), sem_r=("r", "sem"), min_p_n_bins=("p_n_bins", "min"),
    n_sig_n_bins=("p_n_bins", lambda p: int((p < 0.05).sum())), min_p_n_weights=("p_n_weights", "min"),
    n_sig_n_weights=("p_n_weights", lambda p: int((p < 0.05).sum())), tests=("r", "size")).round(4).to_string())
print("\nPer ensemble and frame, 10 degree Idir bins, rotating-trial probabilities:")
print(s3[(s3.bins == 10) & (s3.probabilities == "rotating")].round(4).to_string(index=False))

**Result: reproduced, resting on an assumption.** With 10 degree Idir bins and probabilities
from the rotating trial, the average Rayleigh r is 0.166 (expected 0.17 ± 0.02). With n the sum
of the weights, none of the 14 tests is significant (smallest p = 0.35); with n the number of
bins, one is (1121R157, room frame, p = 0.014). The other Idir readings give average r of 0.180
to 0.264 and 1 to 5 significant tests of 14. The outcome rests on taking n as the sum of the
weights and on the 10 degree, rotating-trial Idir reading.

## fig3-hd-pair-correlations-preserved

> The sub-second temporal coupling of head-direction cell pairs (Kendall correlations of spike
> counts at 40, 250 and 1000 ms) was maintained across stable1, rotating and stable2.

Spike counts in 40, 250 and 1000 ms bins over each trial (from the spike times), Kendall's tau
for each simultaneously recorded pair, Fisher z-transformed; one-way ANOVA of the three
conditions at each timescale. A claim of absence: reproduced when p ≥ 0.05 at all three.

In [ ]:
PAIRS_HD = [(us[i], us[j]) for s, us in SESS.items() for i in range(len(us)) for j in range(i + 1, len(us))]
KT = []
for a, b in PAIRS_HD:
    for tr in ["SA1", "RA_rm", "SA2"]:
        t_end = min(len(data[a][tr]["d"]) / data[a][tr]["fs"], len(data[b][tr]["d"]) / data[b][tr]["fs"])
        for bs in [0.04, 0.25, 1.0]:
            ca = hd20.spike_count_series(data[a][tr], bs, t_end)
            cb = hd20.spike_count_series(data[b][tr], bs, t_end)
            tau = stats.kendalltau(ca, cb).statistic if ca.any() and cb.any() else np.nan
            KT.append({"pair": f"{a}|{b}", "trial": tr, "bin_ms": int(bs * 1000), "tau": tau})
KT = pd.DataFrame(KT)
KT["z"] = np.arctanh(KT.tau)

In [ ]:
print("Pairs:", len(PAIRS_HD), "; pairs with an undefined tau (a cell without spikes):",
      KT[KT.tau.isna()].pair.nunique())
ft = pd.DataFrame([{"session": s, "trial": tr, "units": len(us), "first timestamps": len({data[u][tr]["first_ts"] for u in us})}
                   for s, us in SESS.items() for tr in ["SA1", "RA_rm", "SA2"]])
print("Session trials whose units differ in first timestamp (counts are binned from each unit's own):",
      int((ft["first timestamps"] > 1).sum()), "of", len(ft))
print("Units of session 1017R035 (whose stable2 header states 2000 samples/s and a first timestamp of 5):",
      SESS["1017R035"], "; its pairs:", [p_ for p_ in PAIRS_HD if "1017R035" in p_[0]])
for bs in [40, 250, 1000]:
    k = KT[KT.bin_ms == bs].pivot(index="pair", columns="trial", values="z")[["SA1", "RA_rm", "SA2"]]
    r = hd20.oneway({c: k[c] for c in k.columns})
    rm = hd20.rm_anova(k)
    r1 = stats.pearsonr(*k[["SA1", "RA_rm"]].dropna().T.values)[0]
    r2 = stats.pearsonr(*k[["SA1", "SA2"]].dropna().T.values)[0]
    print(f"{bs} ms: mean z {k.mean().round(4).to_dict()}; one-way ANOVA F({r['df1']}, {r['df2']}) = {r['F']:.3f}, "
          f"p = {r['p']:.3g}; RM ANOVA (variant) {hd20.fmt_anova(rm)}; Pearson r of tau stable1 vs rotating {r1:.3f}, "
          f"vs stable2 {r2:.3f}")
k = KT[KT.bin_ms == 250].pivot(index="pair", columns="trial", values="z")
k_rat = k.assign(rat=[p.split("|")[0].split("-")[0][4:] for p in k.index])
print("\nBy rat, mean z at 250 ms (pairs):")
print(k_rat.groupby("rat")[["SA1", "RA_rm", "SA2"]].mean().round(3).join(k_rat.groupby("rat").size().rename("pairs")).to_string())
print("z range (all timescales): %.3f to %.3f" % (KT.z.min(), KT.z.max()))
fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].scatter(k.SA1, k.RA_rm, s=6)
ax[0].set_xlabel("z stable1, 250 ms")
ax[0].set_ylabel("z rotating")
ax[1].scatter(k.SA1, k.SA2, s=6)
ax[1].set_xlabel("z stable1, 250 ms")
ax[1].set_ylabel("z stable2")
plt.tight_layout()
plt.show()

**Result: reproduced.** The 146 simultaneously recorded head-direction pairs give one-way
ANOVAs of F(2, 435) = 0.098, p = 0.907 at 40 ms, F = 0.329, p = 0.72 at 250 ms and F = 0.483, p =
0.617 at 1000 ms (the paper F(2, 420) = 0.097, 0.305 and 0.44). The units of every session share
their first timestamp in every trial (0 of 132 session trials differ), so the counts of a pair are
binned on a common origin; session 1017R035, whose stable2 header states a first timestamp of 5,
holds one unit and forms no pair. The tau of a pair in stable1 correlates with its tau in the
rotating trial at r = 0.85 to 0.97. A repeated-measures ANOVA (variant) finds small differences at
250 and 1000 ms (p = 0.034 and 0.036), with stable2 lowest.

## figs4, figs5 and figs6

Blocked, data missing. Figure S4 uses all 556 MEC cell pairs and Figures S5 and S6 the 921 MEC
cells modelled with the Gaussian process; the data set holds only the 115 head-direction cells
(data check and cell-counts).

## no-angular-velocity-tuning

> The MEC head-direction cells showed no systematic tuning of firing rate to angular head
> velocity.

Angular velocity from the direction change between successive tracked samples (wrapped,
degrees/s). Per unit, Pearson r of the spike count per sample with (a) signed and (b) unsigned
angular velocity, in stable1 (the classification trial; other trials as variants). Reproduced
when the fraction of units significant at p < 0.05 does not exceed chance by a binomial test at
p < 0.05 (an assumption of the claims file). The parametric p-value treats successive samples
as independent; as variants, the same r is tested against circular shifts of the spike counts by
at least 30 s, and again after removing the head-direction bin means from both series.

In [ ]:
rows = []
for tr in ["SA1", "RA_rm", "SA2"]:
    for u in U:
        x = data[u][tr]
        d = x["d"].astype(float)
        ok = (d[1:] <= 360) & (d[:-1] <= 360)
        av = ((d[1:] - d[:-1] + 180) % 360 - 180) * x["fs"]
        n = x["n"][1:].astype(float)
        av, n = av[ok], n[ok]
        ra, pa = stats.pearsonr(n, av)
        rb, pb = stats.pearsonr(n, np.abs(av))
        rows.append({"trial": tr, "unit": u, "r_signed": ra, "p_signed": pa, "r_unsigned": rb, "p_unsigned": pb})
av_t = pd.DataFrame(rows)
for tr, g in av_t.groupby("trial", sort=False):
    out = []
    for k in ["signed", "unsigned"]:
        nsig = int((g[f"p_{k}"] < 0.05).sum())
        pb = stats.binomtest(nsig, len(g), 0.05, alternative="greater").pvalue
        out.append(f"{k}: {nsig}/{len(g)} significant (binomial p = {pb:.2g}), median r {g[f'r_{k}'].median():.4f}, "
                   f"r range {g[f'r_{k}'].min():.3f} to {g[f'r_{k}'].max():.3f}, positive {int((g[f'r_{k}'] > 0).sum())}")
    print(tr, "|", " | ".join(out))
g = av_t[av_t.trial == "SA1"]
print("\nStable1, |r| above 0.05:", int((g.r_signed.abs() > 0.05).sum()), "signed,", int((g.r_unsigned.abs() > 0.05).sum()), "unsigned")
print("Angular velocity range (deg/s) in one unit's stable1: %.0f to %.0f" % (av.min(), av.max()))

# The parametric p-values above treat the ~54,000 samples as independent. Variant: the same Pearson r
# tested against circular shifts of the spike counts relative to angular velocity by at least 30 s
# (all such shifts, computed by FFT), which keeps the serial dependence of both series; and the same
# shift test after removing from both the mean in each 10 degree head-direction bin, so that a link of
# both to head direction cannot produce a correlation.
def shift_r(a, b):
    a, b = a - a.mean(), b - b.mean()
    c = np.fft.irfft(np.fft.rfft(a) * np.conj(np.fft.rfft(b)), len(a))
    return c / np.sqrt((a * a).sum() * (b * b).sum())


def by_bin(v, hb):
    return v - (np.bincount(hb, weights=v, minlength=36) / np.maximum(np.bincount(hb, minlength=36), 1))[hb]


rows = []
for u in U:
    x = data[u]["SA1"]
    d = x["d"].astype(float)
    ok = (d[1:] <= 360) & (d[:-1] <= 360)
    av = (((d[1:] - d[:-1] + 180) % 360 - 180) * x["fs"])[ok]
    n = x["n"][1:].astype(float)[ok]
    hb = (d[1:][ok] // 10).astype(int) % 36
    m = int(30 * x["fs"])
    for k, v in [("signed", av), ("unsigned", np.abs(av))]:
        for test, (a, b) in [("shift", (n, v)), ("shift, head direction removed", (by_bin(n, hb), by_bin(v, hb)))]:
            c = shift_r(a, b)
            null = np.abs(c[m:len(c) - m])
            rows.append({"unit": u, "velocity": k, "test": test, "r": c[0],
                         "p": (1 + (null >= abs(c[0])).sum()) / (1 + len(null))})
av_s = pd.DataFrame(rows)
print("\nStable1, shift tests (variants):")
for (k, test), g in av_s.groupby(["velocity", "test"]):
    nsig = int((g.p < 0.05).sum())
    print(f"  {k}, {test}: {nsig}/{len(g)} significant (binomial p = {stats.binomtest(nsig, len(g), 0.05, alternative='greater').pvalue:.2g}), "
          f"|r| at most {g.r.abs().max():.3f}, positive {int((g.r > 0).sum())}")


**Result: not reproduced, resting on the claims file's assumed test.** In stable1, 12 of 115
units correlate with signed angular velocity at p < 0.05 (binomial p = 0.013 against 5%) and 49
of 115 with unsigned angular velocity (binomial p = 5.7e-33); the other trials give the same.
Testing the same correlations against circular shifts of at least 30 s, which keeps the serial
dependence of the samples, gives 11 of 115 (binomial p = 0.029) and 41 of 115 (p = 3e-24), and
after removing the head-direction bin means from both series 14 of 115 (p = 0.0019) and 34 of 115
(p = 1.8e-17), so the excess does not come from treating the samples as independent. The
correlations are small (|r| at most 0.029 signed and 0.068 unsigned), of both signs (49 to 53 of
115 positive for unsigned), and the angular velocities reach about 5400 degrees/s, which points
to tracking noise in the direction. The criterion counts significant cells; the sizes of the
correlations do not show tuning.

## fig4-internal-tuning-preserved

> Internally referenced directional tuning of head-direction cell pairs (tuning of one cell to
> head-direction displacement since the other's spikes) was preserved across stable1, rotating
> and stable2; neither its instability nor its change in preference differed between trials or
> from the two halves of stable1.

Internally referenced tuning as Spike_triggered_FR_curve.py computes it (helper
`internal_tuning`), for each simultaneously recorded pair with the first-listed unit of the pair
as cell 1 (the code labels pair i, j with i < j). Readings of the criterion: interval 5 s (main)
or 30 s; cell 2 spikes before the cell 1 spike (the code; main) or after; normalisation (a) the
Methods' division by the sum of the time offsets, a single number per curve, so the curve is the
spike-difference histogram (main), (b) the code's division by the occupancy of each
displacement; a variant, (b rescaled), scales the (b) curve to sum to its spike count, as the
Krichevsky-Trofimov 0.5 otherwise swamps values far below 1. KLD and preferred-direction change between
stable1 and stable2, the rotating room frame and the rotating arena frame, and between the halves
of stable1. Pairs used: those with a defined curve (at least one cell 2 spike in the window) in
every condition. Four tests: RM ANOVA over the three trial pairs for KLD and for preference
change, and one-way ANOVA of the four groups for each. A claim of absence: reproduced when all
four have p ≥ 0.05 under the main readings.

In [ ]:
INTERVALS = [3, 4, 5, 10, 20, 30, 40, 50, 60, 300, 600]
t0 = time.time()
IT = {}
for a, b in PAIRS_HD:
    for side in ["before", "after"]:
        for cond in ["SA1", "RA_rm", "RA_ar", "SA2", "half1", "half2"]:
            tr = cond if not cond.startswith("half") else "SA1"
            x, y = data[a][tr], data[b][tr]
            n = min(len(x["d"]), len(y["d"]))
            sl = slice(0, n)
            if cond == "half1":
                sl = slice(0, n // 2)
            elif cond == "half2":
                sl = slice(n // 2, n)
            wins = INTERVALS if not cond.startswith("half") else [5, 30]
            IT[(a, b, side, cond)] = hd20.internal_tuning(x["d"][sl], x["n"][sl], y["n"][sl], x["fs"], wins, side)
print(f"internally referenced tuning of {len(PAIRS_HD)} pairs computed in {time.time() - t0:.0f} s")


def it_curve(res, T, norm):
    c, o = res[T]["counts"], res[T]["occ"]
    if norm == "a":
        return c
    with np.errstate(invalid="ignore", divide="ignore"):
        b = np.where(o > 0, c / o, 0.0)
    if norm == "b":
        return b
    # variant (b rescaled): the occupancy-normalised curve scaled to sum to its spike count, so that
    # the Krichevsky-Trofimov 0.5 is added on the scale of counts, as the Methods' n_i are
    return b * c.sum() / b.sum() if b.sum() > 0 else b


IT_CENTRES = np.deg2rad(np.arange(36) * 10 - 180 + 5)

In [ ]:
fig4_rows = []
for T in [5, 30]:
    for side in ["before", "after"]:
        for norm in ["a", "b", "b rescaled"]:
            kl, pc, used = {}, {}, 0
            for a, b in PAIRS_HD:
                cv = {c: it_curve(IT[(a, b, side, c)], T, norm) for c in ["SA1", "RA_rm", "RA_ar", "SA2", "half1", "half2"]}
                cnt = {c: IT[(a, b, side, c)][T]["counts"].sum() for c in cv}
                if min(cnt.values()) == 0:
                    continue
                used += 1
                key = f"{a}|{b}"
                kl[key] = {"s1-s2": hd20.kld(cv["SA1"], cv["SA2"]), "s1-room": hd20.kld(cv["SA1"], cv["RA_rm"]),
                           "s1-arena": hd20.kld(cv["SA1"], cv["RA_ar"]), "halves": hd20.kld(cv["half1"], cv["half2"])}
                pk = {c: np.argmax(cv[c]) * 10 for c in cv}
                pc[key] = {"s1-s2": hd20.circ_absdiff(pk["SA1"], pk["SA2"]), "s1-room": hd20.circ_absdiff(pk["SA1"], pk["RA_rm"]),
                           "s1-arena": hd20.circ_absdiff(pk["SA1"], pk["RA_ar"]), "halves": hd20.circ_absdiff(pk["half1"], pk["half2"])}
            for mname, m in [("instability (KLD)", pd.DataFrame(kl).T), ("preference change", pd.DataFrame(pc).T.astype(float))]:
                r1 = hd20.rm_anova(m[["s1-s2", "s1-room", "s1-arena"]])
                r2 = hd20.oneway({c: m[c] for c in m.columns})
                fig4_rows.append({"T_s": T, "side": side, "norm": norm, "measure": mname, "pairs": used,
                                  "means": " ".join(f"{k} {v:.3g}" for k, v in m.mean().items()),
                                  "RM F": f"F({r1['df1']:.2f}, {r1['df2']:.2f}) = {r1['F']:.2f}", "RM p": r1["p"],
                                  "4-group F": f"F({r2['df1']}, {r2['df2']}) = {r2['F']:.2f}", "4-group p": r2["p"]})
f4 = pd.DataFrame(fig4_rows)
pd.set_option("display.max_colwidth", 120)
print(f4.to_string(index=False, formatters={"RM p": "{:.3g}".format, "4-group p": "{:.3g}".format}))
f4["all p >= 0.05"] = (f4["RM p"] >= 0.05) & (f4["4-group p"] >= 0.05)
print("\nReadings (interval, side, normalisation) with all four tests at p >= 0.05:")
print(f4.groupby(["T_s", "side", "norm"])["all p >= 0.05"].all().to_string())
main_kl = {}
for a, b in PAIRS_HD:
    cv = {c: it_curve(IT[(a, b, "before", c)], 5, "a") for c in ["SA1", "RA_rm", "RA_ar", "SA2", "half1", "half2"]}
    if min(v.sum() for v in cv.values()) == 0:
        continue
    main_kl[f"{a}|{b}"] = {"rat": a.split("-")[0][4:], "s1-s2": hd20.kld(cv["SA1"], cv["SA2"]),
                           "s1-room": hd20.kld(cv["SA1"], cv["RA_rm"]), "s1-arena": hd20.kld(cv["SA1"], cv["RA_ar"]),
                           "halves": hd20.kld(cv["half1"], cv["half2"])}
main_kl = pd.DataFrame(main_kl).T
print("\nBy rat, mean instability (KLD), main readings:")
print(main_kl.groupby("rat")[["s1-s2", "s1-room", "s1-arena", "halves"]].mean().astype(float).round(3)
      .join(main_kl.groupby("rat").size().rename("pairs")).to_string())
print("Pairs left out (no cell 2 spike in some condition's windows), 5 s before:",
      len(PAIRS_HD) - int(f4[(f4.T_s == 5) & (f4.side == "before")].pairs.iloc[0]))

**Result: not reproduced.** Under the main readings (5 s, cell 2 spikes before, the Methods'
normalisation; 142 pairs with a defined curve in every condition) instability differs across the
trial pairs (F(2.00, 282.00) = 10.20, p = 5.3e-05) and in the four-group ANOVA (F(3, 564) = 3.76,
p = 0.011), and the preference change differs across trial pairs (F = 6.39, p = 0.0019); only
the four-group ANOVA of the preference change has p ≥ 0.05 (p = 0.125). The differences are
small (mean KLD 0.148 to 0.207 against 0.166 for the halves). Of the eight readings only one (5
s, before, the code's occupancy normalisation) has all four tests at p ≥ 0.05. Its KLD is near 0
(means about 0.001) because the Krichevsky-Trofimov 0.5 swamps the normalised values; rescaled
to the spike count (variant), that reading still has all four tests at p ≥ 0.05, with the
instability test close to the threshold (p = 0.052).

## fig4c-arena-registration-short-lived

> The strength of internally referenced directional tuning is similar in all conditions and
> frames for intervals up to 10 s, but beyond 10 s it falls in the rotating arena frame (20 to 60
> s; stable1 = stable2 = rotating room > rotating arena) and at 300 to 600 s also in the rotating
> room frame (stable1 = stable2 > rotating room > rotating arena).

Strength is the Rayleigh vector length of the internally referenced curve (claims file), per
pair, condition and maximum interval (3 to 600 s), with cell 2 spikes before the cell 1 spike
(the code) or after it, and normalisation (a) (main) or (b), the readings of
fig4-internal-tuning-preserved. One-way ANOVA of the four conditions at each interval with
Tukey-Kramer HSD (between groups, as the criterion's one-way ANOVA). The last lines check each
part of the criterion under each reading.

In [ ]:
COND4 = {"SA1": "stable1", "SA2": "stable2", "RA_rm": "rotating room", "RA_ar": "rotating arena"}
F4C = []
for side, norm in [("before", "a"), ("before", "b"), ("after", "a"), ("after", "b")]:
    print(f"\nCell 2 spikes {side} the cell 1 spike, normalisation ({norm}):")
    rows = []
    for T in INTERVALS:
        st = {}
        for a, b in PAIRS_HD:
            res = {c: IT[(a, b, side, c)] for c in COND4}
            if min(res[c][T]["counts"].sum() for c in COND4) == 0:
                continue
            st[f"{a}|{b}"] = {COND4[c]: hd20.resultant(it_curve(res[c], T, norm), IT_CENTRES)[0] for c in COND4}
        st = pd.DataFrame(st).T
        r = hd20.oneway({c: st[c] for c in st.columns})
        tk = hd20.tukey(st)
        sig = tk[tk["p-adj"] < 0.05]
        order = "; ".join(f"{g1} vs {g2} {d:+.3f}" for g1, g2, d in zip(sig.group1, sig.group2, sig.meandiff))
        arena_low = all(((tk.group1 == "rotating arena") & (tk.group2 == o) | (tk.group2 == "rotating arena") & (tk.group1 == o))
                        .pipe(lambda m: bool((m & tk.reject & (st[o].mean() > st["rotating arena"].mean())).any()))
                        for o in ["stable1", "stable2", "rotating room"])
        room_low = all(((tk.group1 == "rotating room") & (tk.group2 == o) | (tk.group2 == "rotating room") & (tk.group1 == o))
                       .pipe(lambda m: bool((m & tk.reject & (st[o].mean() > st["rotating room"].mean())).any()))
                       for o in ["stable1", "stable2"])
        F4C.append({"side": side, "norm": norm, "T_s": T, "p": r["p"], "arena below other three": arena_low,
                    "room below both stable": room_low})
        rows.append({"T_s": T, "pairs": len(st), **{c: round(st[c].mean(), 3) for c in st.columns},
                     "F": f"F({r['df1']}, {r['df2']}) = {r['F']:.2f}", "p": r["p"],
                     "Tukey p<0.05 (group2 - group1)": order or "none"})
    t = pd.DataFrame(rows)
    print(t.to_string(index=False, formatters={"p": "{:.3g}".format}))
F4C = pd.DataFrame(F4C)
print("\nCriterion, per reading: ANOVA p >= 0.05 at 3 to 10 s, p < 0.05 beyond, arena frame below the other three")
print("(Tukey p < 0.05) at 20 to 60 s, room frame below both stable trials at 300 and 600 s:")
for (side, norm), g in F4C.groupby(["side", "norm"], sort=False):
    short, long_ = g[g.T_s <= 10], g[g.T_s > 10]
    mid, far = g[g.T_s.between(20, 60)], g[g.T_s >= 300]
    print(f"  {side}, ({norm}): p >= 0.05 up to 10 s {bool((short.p >= 0.05).all())} "
          f"(intervals failing {list(short[short.p < 0.05].T_s)}); p < 0.05 beyond {bool((long_.p < 0.05).all())}; "
          f"arena below others at 20-60 s {bool(mid['arena below other three'].all())} "
          f"(at {list(mid[mid['arena below other three']].T_s)}); room below stable at 300-600 s {bool(far['room below both stable'].all())}; "
          f"arena below others beyond 10 s at {list(long_[long_['arena below other three']].T_s)}")


**Result: not reproduced.** No reading of window side and normalisation gives the claimed
pattern. The arena frame is below all three other conditions at none of 20 to 60 s with the
Methods' normalisation (either side) and only at 60 s (cell 2 before) or at 50 and 60 s (after)
with the code's normalisation; at 20 to 50 s the room frame falls as far as or further than the
arena frame (with the Methods' normalisation, before, 40 s: 0.267 and 0.281 against 0.410 and
0.452). The conditions differ already at 10 s in three readings (p = 0.0086, 0.0024 and 0.0094)
and at 5 s with cell 2 after (p = 0.020); only the code's normalisation with cell 2 before has
no difference up to 10 s (p = 0.096 to 0.50). Every reading differs at every interval beyond 10
s (p below 0.002), and at 300 and 600 s the arena frame is below the three others. The room
frame is below both stable trials at 300 and 600 s with the Methods' normalisation, as reported,
but not at 600 s with the code's. The criterion's partial form needs the arena frame to fall
beyond 10 s; it falls only from 50 or 60 s, after the room frame, so the effect claimed for 20
to 60 s is not found.

## fig5-error-inhomogeneous

> The angular distribution of decoding error is inhomogeneous in the room frame (resultant r =
> 0.589) and the arena frame (r = 0.653) and differs from the distribution of sampled head
> directions, which is itself biased in both frames (r = 0.147 room, r = 0.378 arena).

Decoding error during rotation (decoder of fig2, 20 samples main, 4 samples variant) in each
frame. A reader amended the criterion on 2026-11-13 in answer to a question
(`questions/park-2019.yaml`): it now names three readings of what the error distribution is a
distribution over, (i) the mean absolute error in each 10 degree bin of the rat's head direction,
taken as weights over the bin centres, (ii) the head directions of the windows decoded within 15,
30 or 45 degrees, (iii) the signed error angles, the only reading run before the answer. For each:
the resultant length and Rayleigh test (n the number of decoded windows for (i), as the paper's
z = 236.36 at r = 0.589 implies about 680; n = 36 bins shown as a variant), Kuiper's test against
the sampled head directions of the same windows (for (i), the error weights against the
head-direction occupancy over the same bins, `hd20.kuiper_binned`, an assumption as the paper
does not say how the test was applied), and the resultant length of the sampled head directions.
Pooling readings: (a) all seven ensembles pooled, (b) the exemplar ensemble with the most cells
(four ensembles tie at six cells; each is shown), (c) each ensemble apart, with the median r.

In [ ]:
def circ_r(deg):
    deg = np.asarray(deg, float)
    deg = deg[~np.isnan(deg)]
    return float(abs(np.exp(1j * np.deg2rad(deg)).mean())), len(deg)


def error_distributions(err, hd):
    """The three distribution readings of the criterion, each as (r, n for Rayleigh, Kuiper V, Kuiper p)."""
    out = {}
    b = (hd // 10).astype(int) % 36
    occ = np.bincount(b, minlength=36)
    # (i) mean absolute error per 10 degree head-direction bin, as weights over the bin centres
    mean_err = np.bincount(b, weights=np.abs(err), minlength=36) / np.maximum(occ, 1)
    r, _ = hd20.resultant(mean_err)
    out["(i) error by head direction"] = (r, len(err), *hd20.kuiper_binned(mean_err, occ, len(err), len(err)),
                                          int((occ > 0).sum()))
    # (ii) head directions of windows decoded within 15, 30 or 45 degrees
    for thr in [15, 30, 45]:
        acc = hd[np.abs(err) <= thr]
        r, n = circ_r(acc)
        out[f"(ii) head direction, error <= {thr}"] = (r, n, *hd20.kuiper_two_sample(acc, hd), None)
    # (iii) the signed error angles
    r, n = circ_r(err)
    out["(iii) signed error"] = (r, n, *hd20.kuiper_two_sample(err, hd), None)
    return out


rows = []
for w in [20, 4]:
    for s in list(ENS) + ["pooled"]:
        ss = list(ENS) if s == "pooled" else [s]
        recs = {}
        for tr, fr in [("RA_rm", "room"), ("RA_ar", "arena")]:
            err = np.concatenate([DEC[(w, e, tr)]["err"] for e in ss])
            hd = np.concatenate([DEC[(w, e, tr)]["dir"] for e in ss])
            rh, nh = circ_r(hd)
            ok = ~np.isnan(err) & ~np.isnan(hd)
            for reading, (r, n, v, pk, nbins) in error_distributions(err[ok], hd[ok]).items():
                rec = recs.setdefault(reading, {"window": w, "ensemble": s, "reading": reading})
                rec.update({f"{fr} err r": r, f"{fr} err n": n, f"{fr} err p": hd20.rayleigh_p(r, n),
                            f"{fr} kuiper V": v, f"{fr} kuiper p": pk, f"{fr} hd r": rh, f"{fr} hd n": nh,
                            f"{fr} hd p": hd20.rayleigh_p(rh, nh)})
                if nbins is not None:
                    rec[f"{fr} err p (n = bins)"] = hd20.rayleigh_p(r, nbins)
        rows.extend(recs.values())
f5 = pd.DataFrame(rows)
EXP5 = {"room err r": 0.589, "arena err r": 0.653, "room hd r": 0.147, "arena hd r": 0.378}
f5["r within tolerance"] = [sum(abs(r[k] - v) <= 0.1 * v for k, v in EXP5.items()) for _, r in f5.iterrows()]
f5["tests hold"] = (f5[["room err p", "arena err p", "room kuiper p", "arena kuiper p", "room hd p", "arena hd p"]]
                    < 0.05).all(axis=1)
cols = ["window", "ensemble", "reading", "room err r", "room err n", "room err p", "room kuiper p",
        "arena err r", "arena err n", "arena err p", "arena kuiper p", "room hd r", "room hd p", "arena hd r",
        "arena hd p", "tests hold", "r within tolerance"]
fmt = lambda v: f"{v:.3g}"
for w in [20, 4]:
    print(f"Window of {w} samples ({w / 30.02 * 1000:.0f} ms):")
    print(f5[f5.window == w][cols].drop(columns="window").to_string(index=False, float_format=fmt))
    print()
print("Reading (i) with n the number of occupied bins (36) in place of the number of windows:")
print(f5[f5.reading.str.startswith("(i)")][["window", "ensemble", "room err r", "room err p (n = bins)", "arena err r",
                                             "arena err p (n = bins)"]].to_string(index=False, float_format=fmt))
print("\nReading (c), median over ensembles of r:")
print(f5[f5.ensemble != "pooled"].groupby(["window", "reading"])[list(EXP5)].median().round(3).to_string())
print("\nExpected:", EXP5, "(tolerance 10%)")
best = f5.sort_values(["tests hold", "r within tolerance"], ascending=False)
print("\nCombinations (window, pooling, reading) where every test holds:", int(f5["tests hold"].sum()), "of", len(f5),
      "; with all four r within tolerance:", int((f5["tests hold"] & (f5["r within tolerance"] == 4)).sum()))
print("Most r within tolerance where every test holds:",
      int(f5[f5["tests hold"]]["r within tolerance"].max()) if f5["tests hold"].any() else None)
print(f5.groupby(["window", "tests hold"])["r within tolerance"].value_counts().unstack(fill_value=0).to_string())
# the sample sizes the paper's Rayleigh z values imply (z = n r^2), compared with the counts here
from itertools import combinations
IMPL = {"room error": 236.36 / 0.589**2, "arena error": 289.63 / 0.653**2, "room head direction": 230.6 / 0.147**2,
        "arena head direction": 153.2 / 0.378**2}
print("\nn implied by the paper's z values:", {k: round(v) for k, v in IMPL.items()})
cnt = []
for w in [20, 4]:
    for s in ENS:
        cnt.append({"window": w, "ensemble": s, **{f"{tr} {k}": int(np.sum(~np.isnan(DEC[(w, s, tr)][f]))) for tr in ["RA_rm", "RA_ar"]
                                                    for k, f in [("decoded", "err"), ("tracked", "dir")]}})
cnt = pd.DataFrame(cnt)
print(cnt.to_string(index=False))
near_ = lambda v, t: abs(v - t) <= 0.1 * t
hits = []
for w in [20, 4]:
    c = cnt[cnt.window == w].set_index("ensemble")
    for k in range(1, len(ENS) + 1):
        for sub in combinations(ENS, k):
            tot = c.loc[list(sub)].sum()
            for col, target in [("RA_rm decoded", "room error"), ("RA_ar decoded", "arena error"),
                                ("RA_rm tracked", "room head direction"), ("RA_ar tracked", "arena head direction")]:
                if near_(tot[col], IMPL[target]):
                    hits.append({"window": w, "ensembles": len(sub), "count": col, "n": int(tot[col]), "implied": target})
hits = pd.DataFrame(hits)
print("\nSubsets of the seven ensembles whose summed counts are within 10% of an implied n:")
print(hits.groupby(["window", "count", "implied"]).size().rename("subsets").to_string() if len(hits) else "none")
both = f5[f5["room err n"].apply(lambda v: near_(v, IMPL["room error"])) & f5["arena err n"].apply(lambda v: near_(v, IMPL["arena error"]))]
print("Rows above with both error n within 10% of the implied 681 and 679:", len(both))
print("Rows with the room error n within 10% of 681:")
print(f5[f5["room err n"].apply(lambda v: near_(v, IMPL["room error"]))][["window", "ensemble", "reading", "room err n", "arena err n"]]
      .to_string(index=False))


**Result: partially reproduced.** Every test holds (all four distributions non-uniform and both
Kuiper's tests at p < 0.05) in 75 of the 80 combinations of window, pooling and distribution
reading, but none has all four r within tolerance; the most is two. With the six-cell ensemble
0930R029 (reading (b)) and 20-sample windows, the head directions of windows decoded within 15
degrees (reading (ii)) give a room-frame r of 0.604 (expected 0.589) and those within 30 degrees
0.555, and the signed errors (reading (iii)) 0.560, all within tolerance, with the arena-frame
head-direction r 0.360 (expected 0.378); the arena-frame r under these readings is 0.484, 0.543
and 0.144 (expected 0.653), and the room-frame head-direction r 0.072 (expected 0.147). Reading
(i) gives error r of 0.030 to 0.426 at 20 samples. Pooled (reading (a)) no r is within tolerance
under any distribution reading, and the medians over ensembles (reading (c)) reach at most 0.343
in the room frame and 0.548 in the arena frame (reading (ii), 4 samples). The answer adds readings
(i) and (ii); they bring the arena-frame error r closer to 0.653 but not within tolerance, and the
outcome is the one reading (iii) gave. The paper's z values imply about 681 and 679 windows for
the two error distributions and 10,671 and 1,072 samples for the two head-direction
distributions, a tenfold difference between frames of the same trials. No row has both error
counts within 10% of 681 and 679, and no single ensemble or sum of ensembles matches the
head-direction counts in both frames (35 sums of 20-sample windows match the room frame's
10,671 only), so which windows the paper pooled cannot be recovered from these counts.

## fig5c-error-minima-at-cues

> The room-frame decoding error histogram has strong minima at 110 and 350 degrees in the room
> frame ...

Readings: (a) mean absolute room-frame decoding error as a function of the room-frame head
direction, 10 degree bins; (b) the histogram of the signed room-frame error in 10 degree bins.
The two deepest local minima (circular), located by their bin centres; tolerance 10 degrees.
Under the pooling readings of fig5 and both windows. The decoded direction is the start of the
winning bin, as in the paper's code, which shifts the signed error of reading (b) by about -5
degrees. The chance rate of a match, for two minima placed at random, is printed.

In [ ]:
def two_minima(y):
    y = np.asarray(y, float)
    loc = [i for i in range(36) if y[i] < y[i - 1] and y[i] < y[(i + 1) % 36]]
    depth = {i: min(y[i - 1], y[(i + 1) % 36]) - y[i] for i in loc}
    return sorted(loc, key=lambda i: y[i])[:2], depth


def near(deg, target):
    return float(hd20.circ_absdiff(deg, target)) <= 10


rows = []
for w in [20, 4]:
    for s in list(ENS) + ["pooled"]:
        ss = list(ENS) if s == "pooled" else [s]
        err = np.concatenate([DEC[(w, e, "RA_rm")]["err"] for e in ss])
        hd = np.concatenate([DEC[(w, e, "RA_rm")]["dir"] for e in ss])
        ok = ~np.isnan(err) & ~np.isnan(hd)
        b = (hd[ok] // 10).astype(int) % 36
        ya = np.bincount(b, weights=np.abs(err[ok]), minlength=36) / np.maximum(np.bincount(b, minlength=36), 1)
        yb = np.bincount(((err[ok] % 360) // 10).astype(int) % 36, minlength=36)
        for name, y in [("(a) error by head direction", ya), ("(b) signed error histogram", yb)]:
            mins, _ = two_minima(y)
            starts = [i * 10 for i in mins]
            hit = {t: any(near(m + 5, t) for m in starts) for t in [110, 350]}   # bin centres
            rows.append({"window": w, "ensemble": s, "reading": name, "minima (bin starts)": starts,
                         "near 110": hit[110], "near 350": hit[350]})
f5c = pd.DataFrame(rows)
print(f5c.to_string(index=False))
print("\nRows with both minima near 110 and 350:", int((f5c["near 110"] & f5c["near 350"]).sum()), "of", len(f5c),
      "; with one:", int((f5c["near 110"] ^ f5c["near 350"]).sum()))
# chance: the two minima as two distinct bins drawn at random from the 36
from itertools import permutations
pairs_ = list(permutations(range(36), 2))
h = [(any(near(i * 10 + 5, 110) for i in pr_), any(near(i * 10 + 5, 350) for i in pr_)) for pr_ in pairs_]
p_both = np.mean([a and b for a, b in h])
p_one = np.mean([a != b for a, b in h])
print(f"Chance per row (two random distinct bins): both {p_both:.4f}, exactly one {p_one:.3f}; expected over "
      f"{len(f5c)} rows: both {len(f5c) * p_both:.2f}, one {len(f5c) * p_one:.1f}")

**Result: reproduced under one reading.** With the seven ensembles pooled and 20-sample windows,
the mean absolute room-frame error as a function of room-frame head direction (reading (a)) has
its two deepest minima in the bins starting at 100 and 340 degrees (centres 105 and 345), within
10 degrees of 110 and 350 by their bin centres. This is 1 of 32 combinations of window, pooling
and reading, against 0.20 expected by chance for two minima placed at random; 10 others find one
of the two minima, against 6.6 expected by chance, and the pooled 133 ms variant does not find
them. The match rests on the pooled reading.

## Summary

| Claim | Outcome |
|---|---|
| cell-counts | partially reproduced: 115 head-direction units from 13 rats; the other MEC cells and pairs are not deposited |
| hd-cell-rate | reproduced: 1.39 AP/s in stable1 |
| fig1a-avoidance-learning | blocked: data missing |
| fig1c-speed | reproduced |
| fig1c-lfp-power | blocked: data missing |
| fig1c-firing-rate | reproduced (p = 0.18) |
| fig1c-directionality | reproduced, with Tukey HSD on the repeated-measures error term (assumed); partial with the between-groups variant |
| fig1d-preference-change | partially reproduced: stable1-arena change is the largest, not the smallest |
| fig1d-instability | partially reproduced: stable1-room does not exceed the halves |
| fig2-decoding-stable-vs-rotating | reproduced (assumed majority rule) |
| figs2-frame-preference | reproduced under 1 of 16 readings (assumed) |
| fig2c-proximity-anticorrelation | partially reproduced: negative but weaker (mean r -0.04 to -0.09) under three distance measures |
| figs3-no-head-direction-bias | reproduced (assumed n of the Rayleigh test) |
| fig3-hd-pair-correlations-preserved | reproduced |
| figs4-all-pair-correlations-preserved | blocked: data missing |
| figs5-hd-cells-high-model-information | blocked: data missing |
| figs6-subclass-correlations-preserved | blocked: data missing |
| no-angular-velocity-tuning | not reproduced (assumed test) |
| fig4-internal-tuning-preserved | not reproduced |
| fig4c-arena-registration-short-lived | not reproduced: the room frame falls with or before the arena frame at 20 to 60 s |
| fig5-error-inhomogeneous | partially reproduced |
| fig5c-error-minima-at-cues | reproduced under one reading (assumed) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** Spike width cannot be applied (no waveforms); the
115 deposited units are taken as the head-direction cells. Units are matched across trials by
file name where FileNameList.xlsx is shifted. The decoder and the internally referenced tuning
are ports of the paper's Python code; Idir, relative proximity, the strength of internal tuning
and several test details are not fully specified and are run under the readings the claims file
names. Statistics use Python in place of JMP 12, with the Greenhouse-Geisser correction applied
when Mauchly's test is significant. After repeated-measures ANOVAs, Tukey HSD uses the ANOVA's
repeated-measures error term (an assumption; the between-groups Tukey-Kramer HSD is shown as a
variant); after one-way ANOVAs it is the between-groups Tukey-Kramer HSD.

**What was found about the data set.** It holds the 115 head-direction units only (no other MEC
units, LFP, waveforms or training sessions). FileNameList.xlsx shifts the stable2 files of five
units of session 1121R157 by one row. Two pairs of trials from different days share their
tracking sample for sample (stable2 of 1015R029 and 1025R029; the room-frame rotating trial of
1211R039 and 1212R039). The stable1 file of 1024R157-P9C1 holds the spikes of its rotating
trial. One header states 2000 samples/s; one stable2 file lasts 15 min; one spike train has a
decreasing step; the per-rat counts differ from Table S1 for R029 and R523, though both R523
units are named R523 in all their files; samples without a
direction (coded 4294967295), 4 to 73% of a file, are not documented, nor is the angle
convention of the shock zones. One tracker file is named for the wrong rat.